# SL-CBM Results — CUB with CREAM weights

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── change this to your run folder ──────────────────────────────────────────
RUN_DIR = "/home/dani00003/sl-cbm/outputs/20260722122547/evaluations"
# ────────────────────────────────────────────────────────────────────────────

acc      = torch.load(f"{RUN_DIR}/accuracy.pt",              map_location="cpu")
adi      = torch.load(f"{RUN_DIR}/adi_pack.pt",              map_location="cpu")
nec      = torch.load(f"{RUN_DIR}/nec.pt",                   map_location="cpu")
interv   = torch.load(f"{RUN_DIR}/intervention_taskerror.pt",map_location="cpu")

print("Keys — accuracy:", list(acc.keys()))
print("Keys — adi_pack:", list(adi.keys()))
print("Keys — nec:",      list(nec.keys()))
print("Keys — intervention:", list(interv.keys()))

## 1. Accuracy

In [ ]:
print(f"Test Class Accuracy:   {acc['test_acc']:.2f}%")
print(f"Test Concept Accuracy: {acc['test_concept_acc']:.2f}%")

## 2. Explainability — Average Drop / Increase / Gain (ADI)

In [ ]:
import pandas as pd

adi_table = pd.DataFrame({
    "Level":    ["Concept", "Class"],
    "Avg Drop ↓": [
        round(float(adi["concepts_avg_drop"]) * 100, 2),
        round(float(adi["classes_avg_drop"])  * 100, 2),
    ],
    "Avg Inc ↑": [
        round(float(adi["concepts_avg_inc"]) * 100, 2),
        round(float(adi["classes_avg_inc"])  * 100, 2),
    ],
    "Avg Gain ↑": [
        round(float(adi["concepts_avg_gain"]) * 100, 2),
        round(float(adi["classes_avg_gain"])  * 100, 2),
    ],
})
print(adi_table.to_string(index=False))

## 3. Intervention — NEC / ANEC

In [ ]:
print("NEC / ANEC scores:")
for k, v in nec.items():
    print(f"  {k}: {float(v):.4f}")

## 4. Intervention Curve (task error vs. # concepts corrected)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for method, errors in interv.items():
    if isinstance(errors, torch.Tensor):
        y = errors.numpy()
    else:
        y = np.array(errors)
    ax.plot(range(len(y)), y, marker="o", label=method)

ax.set_xlabel("# Concepts Intervened")
ax.set_ylabel("Task Error")
ax.set_title("SL-CBM Intervention Curve (CUB, CREAM weights)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/intervention_curve.png", dpi=150)
plt.show()

## 5. Summary Table (copy into paper comparison)

In [ ]:
nec5  = float(nec.get("NEC-5",  nec.get("nec_5",  0)))
anec  = float(nec.get("ANEC",   nec.get("anec",   0)))

summary = pd.DataFrame([{
    "Method":          "SL-CBM (CREAM)",
    "Class Acc":       round(float(acc["test_acc"]), 2),
    "Concept Acc":     round(float(acc["test_concept_acc"]), 2),
    "NEC-5":           round(nec5, 2),
    "ANEC":            round(anec, 2),
    "Concept AD↓":     round(float(adi["concepts_avg_drop"]) * 100, 2),
    "Concept AI↑":     round(float(adi["concepts_avg_inc"])  * 100, 2),
    "Concept AG↑":     round(float(adi["concepts_avg_gain"]) * 100, 2),
    "Class AD↓":       round(float(adi["classes_avg_drop"])  * 100, 2),
    "Class AI↑":       round(float(adi["classes_avg_inc"])   * 100, 2),
    "Class AG↑":       round(float(adi["classes_avg_gain"])  * 100, 2),
}])

print(summary.T.to_string(header=False))

## 6. Top-5 Concept Groups by Classifier Weight

In [ ]:
import sys
sys.path.insert(0, "/home/dani00003/sl-cbm")
sys.path.insert(0, "/home/dani00003/pcbm-module")

from utils.constants import CUB_features

CKPT_PATH = "/home/dani00003/sl-cbm/outputs/20260722122547/trainable_weights.pt"
state = torch.load(CKPT_PATH, map_location="cpu")

# Find the classifier weight tensor (shape: [n_classes, n_concepts])
weight_key = next(k for k in state if "weight" in k and "posthoc" in k.lower()
                  or "classifier" in k.lower() and "weight" in k)
W = state[weight_key]  # [n_classes, 112]
print(f"Classifier weight shape: {W.shape}  (key: {weight_key})")

# Mean absolute weight per concept across all classes
mean_abs = W.abs().mean(dim=0)  # [112]

# CUB_features maps group_name -> (start_idx, end_idx)
cub_groups = {k: v for k, v in vars(CUB_features).items()
              if not k.startswith("_") and isinstance(v, tuple)}

group_scores = {}
for group, (start, end) in cub_groups.items():
    group_scores[group] = mean_abs[start:end+1].mean().item()

top5 = sorted(group_scores.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nTop-5 concept groups by mean |weight|:")
for rank, (name, score) in enumerate(top5, 1):
    print(f"  {rank}. {name:35s}  score={score:.4f}")

top5_names = [name for name, _ in top5]
print("\nFor submit_interpret.sh:")
print('CONCEPTS=("' + '" "'.join(top5_names) + '")')